In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# 假设你的数据是一个Series
#df_good = pd.read_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/functional_properties_with_python_measurements.pkl')
df_good = pd.read_pickle(r'/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/functional_properties_with_python_measurements_pycirc.pkl')
df_py = df_good[df_good['buzaki_py_cell_type']=='pyramidal']
df = df_py
base_folder = r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/Results/Functional_cell_type"
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']
sessions = ["A"]

# Put this near the top of your script
auto_legend_text = []

for session in sessions:
    # Filter for sessions
    if session == "Total":
        df_a = df
    else:
        df_a = df[df['session'] == session]

    # Separate into control and experimental groups
    control_df = df_a[df_a['animal_id'].isin(control_ids)]
    exp_df = df_a[df_a['animal_id'].isin(exp_ids)]


df_con_deep = control_df[control_df['sub_population']=="deep"].reset_index(drop=True)['Information_content_rate']
df_con_superficial = control_df[control_df['sub_population']=="superficial"].reset_index(drop=True)['Information_content_rate']
df_exp_deep = exp_df[exp_df['sub_population']=="deep"].reset_index(drop=True)['Information_content_rate']
df_exp_superficial = exp_df[exp_df['sub_population']=="superficial"].reset_index(drop=True)['Information_content_rate']

def group_indices_by_step(df_col, step=0.075):
    used_indices = set()
    result = []
    values = df_col.copy()
    max_val = values.max()

    while len(used_indices) < len(values):
        remaining = values[~values.index.isin(used_indices)]
        if remaining.empty:
            break

        current_group = []
        current_val = remaining.min()
        current_idx = remaining.idxmin()
        current_group.append(current_idx)
        used_indices.add(current_idx)

        while True:
            target_val = current_val + step
            remaining = values[~values.index.isin(used_indices)]
            if remaining.empty:
                break

            # 找到大于等于 target_val 的值中最接近 target_val 的那个
            diffs = remaining - target_val
            diffs = diffs[diffs >= 0]
            if diffs.empty:
                break

            next_idx = diffs.idxmin()
            current_val = values[next_idx]
            current_group.append(next_idx)
            used_indices.add(next_idx)

        result.append(current_group)

    return result
results_con_deep = group_indices_by_step(df_con_deep)
results_con_superficial = group_indices_by_step(df_con_superficial)
results_exp_deep = group_indices_by_step(df_exp_deep)
results_exp_superficial = group_indices_by_step(df_exp_superficial)

def plot_rate_map_panel(ax, results_con, df, sublayer="deep", group = "control", map_color='jet',y_limit=5):
    i_row = 0
    base_y = 0.35
    y_step = 0.5

    for indices in results_con:
        y_coord = base_y + (i_row * y_step)

        if len(indices) > 0:
            values = [df[i] for i in indices]
            num_ratemaps = len(indices)
            for i, (idx, value) in enumerate(zip(indices, values)):
                img_path = fr"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/Results/rate map histogram/{group}/{sublayer}/{map_color}/{idx}.png"
                plot_image_at_xy(ax, img_path, value, y_coord, max_size=0.014)
        i_row += 1

    ax.set_xlim(0, 4.5)
    ax.set_ylim(0, y_limit)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='x', labelsize=10.5)
    ax.set_xlabel("Information Content Rate (spikes / bit)", fontsize=10.5)
    ax.set_yticks([])
    ax.vlines(x=[1, 2, 3, 4], ymin=0, ymax=15, colors='grey', linestyles='dashed', linewidth=3)
    
# Function to load and plot an image with dynamic size adjustment
def plot_image_at_xy(ax, img_path, x, y, max_size=0.09):
    if os.path.exists(img_path):
        img = plt.imread(img_path)
        img_height, img_width = img.shape[:2]
        aspect_ratio = img_width / img_height

        # Calculate zoom based on max_size and aspect ratio
        trans = ax.transData
        fig = ax.get_figure()
        dpi = fig.dpi
        xlim = ax.get_xlim()
        data_width = xlim[1] - xlim[0]
        bbox = ax.get_window_extent().transformed(fig.dpi_scale_trans.inverted())
        axes_width_pixels = bbox.width * dpi
        pixels_per_data = axes_width_pixels / data_width
        size_pixels = max_size * pixels_per_data
        # zoom = size_pixels / max(img_width, img_height / aspect_ratio)

        zoom = 0.017  # Adjust this value to control ratemap size
        imagebox = OffsetImage(img, zoom=zoom)
        ab = AnnotationBbox(imagebox, (x, y), frameon=False, pad=0)
        ax.add_artist(ab)
        return ab
    else:
        print(f"Image {img_path} not found")
        return None

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, shapiro, ttest_ind, mannwhitneyu
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np

# Data loading
folder_path = r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/file_with_table/ripple_ch"

def get_pkl_files(folder_path):
    all_files = os.listdir(folder_path)
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']
pkl_files = get_pkl_files(folder_path)

# Variables to analyze
variables = ['Information_content_rate', 'Sparsity', 'Field_size', 'Averate_rate', 'Selectivity', 'stability_ma']
titles = ['Information content rate\n (spikes/bit)', 'Sparsity', 'Max Field Size', 'Firing Rate (Hz)', 'Selectivity', 'Stability']

# metrics = ['Information_content_rate', 'Sparsity', 'Selectivity', 'Field_size', 'matlab_stability_smooth1']
# titles = ['Information content rate\n (spikes/bit)', 'Sparsity', 'Selectivity', 'Field size', 'stability']


all_dfs = []

# Process each pickle file
for file in pkl_files:
    try:
        df = pd.read_pickle(os.path.join(folder_path, file))
    except Exception as e:
        continue
    #df = df[(df['cell_type'] == "pyramidal") & (df['session'] == "A")]

    df = df[(df['buzaki_py_cell_type'] == "pyramidal") & (df['session'] == "A")]
    #df = df[df['session'] == "A"]
    if df.empty:
        print(f"Warning: File {file} has no pyramidal cells")
        continue
    try:
        animal_id = df['animal_id'].iloc[0]
    except KeyError:
        print(f"Warning: File {file} does not have 'animal_id' column")
        continue
    if any(animal_id.startswith(prefix) for prefix in target_prefixes_control):
        df['group_ani'] = 'control'
    elif any(animal_id.startswith(prefix) for prefix in target_prefixes_exp):
        df['group_ani'] = 'exp'
    else:
        print(f"Warning: animal_id {animal_id} does not match any group")
        continue
    all_dfs.append(df)

y_pos = "addjust y r2"
combined_df=[]
combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df['depth'] = combined_df[y_pos].apply(lambda x: 'deep' if x > 0 else 'superficial')
combined_df['group_depth'] = combined_df['group_ani'] + '_' + combined_df['depth']

# Convert variables to numeric and handle invalid values
for var in variables:
    combined_df[var] = pd.to_numeric(combined_df[var], errors='coerce')
    if combined_df[var].isna().any():
        print(f"Warning: {var} contains NaN values after conversion to numeric")

# Function to test normality and choose test
def choose_stat_test(data1, data2, var_name, group1_name, group2_name):
    # Drop NaN values and ensure numeric
    data1 = data1.dropna()
    data2 = data2.dropna()
    
    # Check for non-numeric values
    if not np.issubdtype(data1.dtype, np.number) or not np.issubdtype(data2.dtype, np.number):
        print(f"Error: Non-numeric data detected in {var_name} for {group1_name} or {group2_name}")
        print(f"{group1_name} dtype: {data1.dtype}, {group2_name} dtype: {data2.dtype}")
        print(f"{group1_name} sample: {data1.head()}")
        print(f"{group2_name} sample: {data2.head()}")
        return "Invalid", np.nan, np.nan

    # Check if data is empty after dropping NaNs
    if len(data1) == 0 or len(data2) == 0:
        print(f"Error: Empty dataset for {var_name} in {group1_name} or {group2_name} after dropping NaNs")
        return "Empty", np.nan, np.nan

    # Perform Shapiro-Wilk test for normality
    stat1, p1 = shapiro(data1)
    stat2, p2 = shapiro(data2)
    
    print(f"{var_name} - {group1_name} Shapiro-Wilk: p={p1:.4f}")
    print(f"{var_name} - {group2_name} Shapiro-Wilk: p={p2:.4f}")
    
    # Choose test based on normality
    if p1 > 0.05 and p2 > 0.05:  # Both are normal
        print(f"{var_name} - Using t-test (both groups normal)")
        stat, p = ttest_ind(data1, data2, equal_var=True)  # Assuming equal variances
        test_name = "t-test"
    else:
        print(f"{var_name} - Using Mann-Whitney U test (non-normal distribution)")
        stat, p = mannwhitneyu(data1, data2, alternative='two-sided')
        test_name = "Mann-Whitney U"
    
    return test_name, stat, p

group_ani = []
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']
for index, row in combined_df.iterrows():
    if str(row['matlab_animal']) in control_ids:
        group_ani.append("control")
    else:
        group_ani.append("exp")
combined_df['group_ani']=None
combined_df['group_ani']=group_ani

from scipy.stats import binomtest
control_df_deep = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['sub_population'] == 'deep')]
exp_df_deep = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['sub_population'] == 'deep')]

# # Control_superficial vs. Exp_superficial
# control_superficial = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['depth'] == 'superficial')][var]
# exp_superficial = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['depth'] == 'superficial')][var]

control_df_place = control_df_deep[(control_df_deep['h_0_place_cell'] == 1) & 
                                (control_df_deep['Information_content_rate'] >= 1.68) & 
                                (control_df_deep['matlab_maxfsize'] >= 20) ]
exp_df_place = exp_df_deep[(exp_df_deep['h_0_place_cell'] == 1) & 
                        (exp_df_deep['Information_content_rate'] >= 1.68) & 
                        (exp_df_deep['matlab_maxfsize'] >= 20) ]
# Calculate numbers for pie charts
control_place_deep = len(control_df_place)
control_non_place_deep = len(control_df_deep) - control_place_deep
exp_place_deep = len(exp_df_place)
exp_non_place_deep = len(exp_df_deep) - exp_place_deep
result = binomtest(exp_place_deep, len(exp_df_deep), p=(control_place_deep /len(control_df_deep)), alternative='less')
p_value_deep = result.pvalue

from scipy.stats import binomtest
control_df_superficial = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['sub_population'] == 'superficial')]
exp_df_superficial = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['sub_population'] == 'superficial')]

# # Control_superficial vs. Exp_superficial
# control_superficial = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['depth'] == 'superficial')][var]
# exp_superficial = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['depth'] == 'superficial')][var]

control_df_place_superficial = control_df_superficial[(control_df_superficial['h_0_place_cell'] == 1) & 
                                (control_df_superficial['Information_content_rate'] >= 1.68) & 
                                (control_df_superficial['matlab_maxfsize'] >= 20)]
exp_df_place_superficial = exp_df_superficial[(exp_df_superficial['h_0_place_cell'] == 1) & 
                        (exp_df_superficial['Information_content_rate'] >= 1.68) & 
                        (exp_df_superficial['matlab_maxfsize'] >= 20) ]

# Calculate numbers for pie charts
control_place_superficial = len(control_df_place_superficial)
control_non_place_superficial = len(control_df_superficial) - control_place_superficial
exp_place_superficial = len(exp_df_place_superficial)
exp_non_place_superficial = len(exp_df_superficial) - exp_place_superficial
result = binomtest(exp_place_superficial, len(exp_df_superficial), p=(control_place_superficial /len(control_df_superficial)), alternative='less')
p_value_superficial = result.pvalue


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.collections import LineCollection
from probeinterface import Probe, ProbeGroup
import probeinterface as pi
from probeinterface.plotting import plot_probe
from scipy.stats import ks_2samp # 确保你的其他统计库已经导入
from scipy.stats import chi2_contingency

# ==========================================
# 1. CONFIGURATION & STYLE (Neuron 级别高级排版)
# ==========================================
FONT_SIZE = 7
TEXT_KWARGS = {'fontsize': FONT_SIZE, 'color': 'black'}
PALETTE = {'control': 'blue', 'exp': 'red'}  # 严格保持原始红蓝配色

fig = plt.figure(figsize=(7.2, 11), dpi=1200)

plt.rcParams.update({
    'font.size': FONT_SIZE,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'sans-serif'],
    'axes.labelsize': FONT_SIZE,
    'axes.titlesize': FONT_SIZE,
    'xtick.labelsize': FONT_SIZE,
    'ytick.labelsize': FONT_SIZE,
    'legend.fontsize': FONT_SIZE,
    'axes.linewidth': 0.8,       # 坐标轴边框加粗，显得扎实
    'xtick.major.width': 0.8,    
    'ytick.major.width': 0.8,
    'xtick.direction': 'out',    
    'ytick.direction': 'out',
    'axes.spines.top': False,    
    'axes.spines.right': False,
    'axes.labelpad': 5,
    'ytick.major.pad': 2,
    'xtick.major.pad': 1,        # 稍微拉近一点刻度数字与坐标轴的距离
    'ytick.major.size': 3,
    'xtick.major.size': 3,
    'pdf.fonttype': 42,          # 保证导出 PDF 时字体可编辑
    'ps.fonttype': 42
})

gs = gridspec.GridSpec(7, 8, height_ratios=[1.5,1,1,1,1,1,1], width_ratios=[1, 1, 1, 1, 1, 1, 1, 1])

# ==========================================
# 2. 绘制辅助函数 (加入背景散点与精细调整)
# ==========================================
# ==========================================
# 2. 绘制辅助函数 (加入背景散点与精细调整)
# ==========================================
def _mixedlm_p(d, y_col):
    """LMM p for genotype with animal as random intercept (right-skewed metrics logged)."""
    import statsmodels.formula.api as smf
    import warnings as _w
    LOGV = {'Information_content_rate', 'Field_size', 'Averate_rate', 'Selectivity'}
    dd = d[['animal_id', 'group_ani', y_col]].copy()
    dd['y'] = pd.to_numeric(dd[y_col], errors='coerce')
    dd = dd.dropna(subset=['y'])
    if y_col in LOGV:
        dd = dd[dd['y'] > 0]; dd['y'] = np.log(dd['y'])
    dd['g'] = pd.Categorical(dd['group_ani'], categories=['control', 'exp'])
    with _w.catch_warnings():
        _w.simplefilter('ignore')
        try:
            return smf.mixedlm("y ~ g", dd, groups=dd['animal_id']).fit(reml=True).pvalues.get('g[T.exp]', np.nan)
        except Exception:
            return np.nan


def _gee_place_p(cdf, layer):
    """Cluster-robust logistic (GEE, cluster=animal) p for place-cell proportion."""
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    import warnings as _w
    d = cdf[cdf['sub_population'] == layer].copy()
    d['is_place'] = ((d['h_0_place_cell'] == 1)
                     & (pd.to_numeric(d['Information_content_rate'], errors='coerce') >= 1.68)
                     & (pd.to_numeric(d['matlab_maxfsize'], errors='coerce') >= 20)).astype(int)
    d['g'] = pd.Categorical(d['group_ani'], categories=['control', 'exp'])
    d = d.dropna(subset=['is_place'])
    with _w.catch_warnings():
        _w.simplefilter('ignore')
        try:
            m = smf.gee("is_place ~ g", "animal_id", d, family=sm.families.Binomial(),
                        cov_struct=sm.cov_struct.Exchangeable()).fit()
            return m.pvalues.get('g[T.exp]', np.nan)
        except Exception:
            return np.nan


def plot_elegant_comparison(ax, df, x_col, y_col, p_val, title):
    """SuperPlot: faint violin (cell distribution) + cells colored by animal +
    per-animal mean (large dots) + mean +/- SEM across mice.
    Significance = linear mixed model with animal as random intercept (NOT the
    cell-level t-test in p_val), so the figure matches the reported statistics
    and exposes the per-animal structure (addresses reviewers on pseudoreplication)."""
    plot_order = ['control', 'exp']
    xmap = {'control': 0, 'exp': 1}
    base = {'control': '#0000FF', 'exp': '#FF0000'}
    d = df.dropna(subset=[y_col]).copy()

    # faint violin = cell-level distribution
    sns.violinplot(data=d, x=x_col, y=y_col, order=plot_order, ax=ax, hue=x_col,
                   palette=base, cut=0, inner=None, linewidth=0, legend=False)
    for coll in ax.collections:
        coll.set_alpha(0.12)

    # cells colored by animal + per-animal mean (large dot)
    for g in plot_order:
        sub = d[d[x_col] == g]
        animals = sorted(sub['animal_id'].unique())
        for ai, a in enumerate(animals):
            yy = pd.to_numeric(sub[sub['animal_id'] == a][y_col], errors='coerce').dropna().values
            if len(yy) == 0:
                continue
            xx = np.random.normal(xmap[g], 0.06, len(yy))
            ax.scatter(xx, yy, s=2.5, color=base[g], alpha=0.30, linewidth=0, zorder=2)
            ax.scatter(xmap[g], np.mean(yy), s=24, color=base[g], edgecolor='k', linewidth=0.6, zorder=4)

    # mean +/- SEM across mice (the experimental unit)
    for g in plot_order:
        am = d[d[x_col] == g].groupby('animal_id')[y_col].mean().values
        if len(am) > 0:
            ax.errorbar(xmap[g], np.mean(am), yerr=np.std(am) / np.sqrt(len(am)),
                        fmt='_', color='k', capsize=3, markersize=10, zorder=5, elinewidth=1.0)

    ax.set_ylabel(title, **TEXT_KWARGS)
    ax.set_xlabel('')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation=-45, ha='left', rotation_mode='anchor', **TEXT_KWARGS)

    # significance bracket from the mixed-effects model
    p_lmm = _mixedlm_p(d, y_col)
    if pd.notna(p_lmm) and p_lmm < 0.05:
        y_max = pd.to_numeric(d[y_col], errors='coerce').max()
        yr = ax.get_ylim()[1] - ax.get_ylim()[0]
        bar_h = yr * 0.03
        sig = '***' if p_lmm < 0.001 else '**' if p_lmm < 0.01 else '*'
        by = [y_max + bar_h, y_max + bar_h * 1.8, y_max + bar_h * 1.8, y_max + bar_h]
        ax.plot([0, 0, 1, 1], by, color='black', lw=0.8)
        ax.text(0.5, y_max + bar_h * 1.5, sig, ha='center', va='center', **TEXT_KWARGS)


def add_bracket_to_bar(ax, x_pos, y_max, p_val):
    """Bracket + explicit p-value annotation for the proportion bars (GEE p)."""
    y_range = ax.get_ylim()[1]
    bar_h = y_range * 0.03
    p_txt = f"p = {p_val:.3g}" if pd.notna(p_val) else "p = n/a"
    if pd.notna(p_val) and p_val < 0.05:
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*'
        bracket_y = [y_max + bar_h, y_max + bar_h*1.8, y_max + bar_h*1.8, y_max + bar_h]
        ax.plot([x_pos[0], x_pos[0], x_pos[1], x_pos[1]], bracket_y, color='black', lw=0.8)
        ax.text((x_pos[0]+x_pos[1])/2, y_max + bar_h*2.8, f"{sig}  {p_txt}", ha='center', va='center', **TEXT_KWARGS)
    else:
        ax.text((x_pos[0]+x_pos[1])/2, y_max + bar_h*2.0, p_txt, ha='center', va='center', **TEXT_KWARGS)


# ==========================================
# Row 0: Signals, Probe, Barplot
# ==========================================
linewidth = 0.6 
ax1_1 = fig.add_subplot(gs[0, 0])
ax1_1.axis('off')

# -- Ripples --
ax1_2 = fig.add_subplot(gs[0, 1])
fs_rip = 1000  
t = np.linspace(0, 0.5, int(fs_rip * 1))  
n_channels = 11  
f_ripple = 150  
noise_level = 0.1  

amplitudes = (np.float32([0.15066703, 0.15041492, 0.15354033, 0.15755153, 0.16833866, 0.17786268, 0.18966555, 0.19084985, 0.18718866, 0.18549542, 0.17947777])*10)**5
amplitudes = amplitudes / np.max(amplitudes) / 0.5
centers = [0.23, 0.27]  

signals = []
for i in range(n_channels):
    signal = np.zeros_like(t)
    center = centers[i % 2]
    amp = amplitudes[i]
    envelope = amp * np.exp(-((t - center) ** 2) / (2 * 0.01 ** 2))
    ripple = envelope * np.sin(2 * np.pi * f_ripple * t)
    noise = noise_level * np.random.randn(len(t))
    signal += ripple + noise
    signals.append(signal)

max_amp_idx = 6
for i in range(n_channels):
    if i == max_amp_idx:
        ax1_2.plot(t, signals[i] + i * 2, color='#FF0000', linewidth=linewidth+0.3, label='Max Amplitude', zorder=5)
    else:
        ax1_2.plot(t, signals[i] + i * 2, color='black', linewidth=linewidth)

ax1_2.invert_yaxis()
ax1_2.axis('off')
ax1_2.set_xlim(.2, .3)

# -- Probe Plot --
ax1_3 = fig.add_subplot(gs[0, 2:6])
probe = pi.get_probe('cambridgeneurotech', 'ASSY-236-F')
df = pd.read_pickle(r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/file_with_table/ripple_ch/63383_2024-07-25_A_units_table_withDLC.pkl")

mapping_to_device = [
    41, 39, 38, 37, 35, 34, 33, 32, 29, 30, 28, 26, 25, 24, 22, 20,
    46, 45, 44, 43, 42, 40, 36, 31, 27, 23, 21, 18, 19, 17, 16, 14,
    55, 53, 54, 52, 51, 50, 49, 48, 47, 15, 13, 12, 11, 9, 10, 8,
    63, 62, 61, 60, 59, 58, 57, 56, 7, 6, 5, 4, 3, 2, 1, 0
]
probe.set_device_channel_indices(mapping_to_device)
probe_df = probe.to_dataframe(complete=True)

probegroup = ProbeGroup()
probegroup.add_probe(probe)
plot_probe(probe, contacts_colors="#D3D3D3", ax=ax1_3) 

df_py_1 = df[df['addjust y r2'] > 0] 
sns.scatterplot(x=df_py_1['x'], y=df_py_1['y'], s=35, alpha=0.9, marker='^', edgecolor='orange', facecolor='none', linewidth=1.0, ax=ax1_3, zorder=3)
df_py_2 = df[df['addjust y r2'] <= 0] 
sns.scatterplot(x=df_py_2['x'], y=df_py_2['y'], s=35, alpha=0.9, marker='^', edgecolor='darkviolet', facecolor='none', linewidth=1.0, ax=ax1_3, zorder=3)

chs = [20, 6, 28, 53, 44]
line_length = 100 
for ch in chs:
    temp = probe_df[probe_df['device_channel_indices'] == ch]
    if not temp.empty:
        x, y = temp['x'].iloc[0], temp['y'].iloc[0]
        ax1_3.plot([x - line_length/2, x + line_length/2], [y, y], 'k--', linewidth=0.6, alpha=0.7)

ax1_3.axis('off')
ax1_3.set_title('')

# -- Position Barplot --
ax1_4 = fig.add_subplot(gs[0, 6:8])
y_pos = "addjust y r2"
sns.barplot(data=combined_df, x='group_ani', y=y_pos, hue='group_depth', ax=ax1_4, width=0.7, errorbar='se', errwidth=1.2, capsize=0)

for patch in ax1_4.patches:
    patch.set_linewidth(1.0)
    try:
        hue = ax1_4.get_legend().get_texts()[ax1_4.patches.index(patch) % len(ax1_4.get_legend().get_texts())].get_text()
        if 'superficial' in hue:
            patch.set_facecolor('#B755E1')  
            patch.set_edgecolor('#0000FF' if 'control' in hue else '#FF0000')
        else:
            patch.set_facecolor('#FFBB41')
            patch.set_edgecolor('#0000FF' if 'control' in hue else '#FF0000')
    except: pass 

ax1_4.set_xlabel('')
ax1_4.set_ylabel('μm')
sns.despine(ax=ax1_4, trim=True, offset=3)
if ax1_4.get_legend(): ax1_4.get_legend().remove()
ax1_4.set_yticks([-20, 0, 20])

# 【强制修改 2】
ax1_4.set_xticks([0, 1])
ax1_4.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation=-45, ha='left', **TEXT_KWARGS)
#plt.setp(ax1_4.get_xticklabels(), rotation=-45, ha='left', rotation_mode='anchor')


# ==========================================
# Row 1, 2, 4, 5: Rate Maps 
# (本地函数保持原样)
# ==========================================
ax2 = fig.add_subplot(gs[1, :])
ax3 = fig.add_subplot(gs[2, :])
ax5 = fig.add_subplot(gs[4, :])
ax6 = fig.add_subplot(gs[5, :])

plot_rate_map_panel(ax2, results_con_deep, df_con_deep, sublayer="deep", y_limit=5.5)
plot_rate_map_panel(ax3, results_exp_deep, df_exp_deep, sublayer="deep", group="exp", y_limit=5.5)
plot_rate_map_panel(ax5, results_con_superficial, df_con_superficial, sublayer="superficial", y_limit=4.5)
plot_rate_map_panel(ax6, results_exp_superficial, df_exp_superficial, sublayer="superficial", group="exp", y_limit=4.5)


# ==========================================
# Row 3: Deep Sub-population Stats
# ==========================================
ax4_1 = fig.add_subplot(gs[3, 0])
ax4_2 = fig.add_subplot(gs[3, 1])
ax4_3 = fig.add_subplot(gs[3, 2])
ax4_4 = fig.add_subplot(gs[3, 3])
ax4_5 = fig.add_subplot(gs[3, 4])
ax4_6 = fig.add_subplot(gs[3, 5])
ax4_7 = fig.add_subplot(gs[3, 6:8])
axis_deep = [ax4_1, ax4_2, ax4_3, ax4_4, ax4_5, ax4_6]

auto_legend_text.append("### Figure X Legend Statistics ###\n")
auto_legend_text.append("Deep Sub-population:")

for i, (var, title) in enumerate(zip(variables, titles)):
    # Add .dropna() to ensure accurate 'n' counts
    control_deep = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['sub_population'] == 'deep')][var].dropna()
    exp_deep = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['sub_population'] == 'deep')][var].dropna()
    
    test_name_deep, stat_deep, p_valued = choose_stat_test(control_deep, exp_deep, title, "Control_deep", "Exp_deep")
    
    # 1. Format the text for Neuron
    n_ctrl = len(control_deep)
    n_exp = len(exp_deep)
    p_str = f"p = {p_valued:.4g}" if p_valued >= 0.0001 else "p < 0.0001"
    
    legend_line = (f"For {title}: {test_name_deep}, statistic = {stat_deep:.3f}, {p_str} "
                   f"(CR;DTA- n = {n_ctrl} cells, CR;DTA+ n = {n_exp} cells).")
    auto_legend_text.append(legend_line)

    subset_df = combined_df[combined_df['depth'] == 'deep'].dropna(subset=[var])
    plot_elegant_comparison(axis_deep[i], subset_df, 'group_ani', var, p_valued, title)

# -- Stacked Bar (Deep) --
ax = ax4_7

contingency_deep = [
    [control_place_deep, control_non_place_deep],
    [exp_place_deep, exp_non_place_deep]
]
chi2_stat_deep, p_value_deep, dof_deep, expected_deep = chi2_contingency(contingency_deep)
print(f"Deep Proportion Chi-Square: stat={chi2_stat_deep:.3f}, p={p_value_deep:.4f}")

# 2. Add to Auto-Legend
p_str_bar = f"p = {p_value_deep:.4g}" if p_value_deep >= 0.0001 else "p < 0.0001"
bar_legend = (f"Proportion of Place vs Non-place cells (Deep): Pearson's Chi-square test, \chi^2 = {chi2_stat_deep:.3f}, {p_str_bar} "
              f"(CR;DTA- n = {len(control_df_deep)} cells, CR;DTA+ n = {len(exp_df_deep)} cells).")
auto_legend_text.append(bar_legend)

groups = ['CR;DTA-', 'CR;DTA+']
counts = [[round(control_place_deep/len(control_df_deep),2)*100, round(control_non_place_deep/len(control_df_deep),2)*100], 
          [round(exp_place_deep/len(exp_df_deep),2)*100, round(exp_non_place_deep/len(exp_df_deep),2)*100]]
percentages = counts 
colors = ['#0000FF', '#B3B3FF'] # 控制组：深蓝和浅蓝
colors1 = ['#FF0000', '#FFB3B3'] # 实验组：深红和浅红

bar_width = 0.4
x_pos = np.arange(len(groups)) * 0.6 

bars = []
for i in range(len(counts[0])):
    bar = ax.bar(x_pos, [counts[j][i] for j in range(len(counts))], 
                 bottom=[sum(counts[j][:i]) for j in range(len(counts))], 
                 color=[colors[i] if j == 0 else colors1[i] for j in range(len(counts))], 
                 width=bar_width, edgecolor='white', linewidth=0.8) 
    bars.append(bar)

for i, bar_group in enumerate(bars):
    cell_type = "Place cell" if i == 0 else "Non\nplace cell"
    for j, bar in enumerate(bar_group):
        ax.text(bar.get_x() + bar.get_width()/2, sum(counts[j][:i]) + bar.get_height()/2, 
                f'{percentages[j][i]:.0f}%\n({cell_type})', 
                ha='center', va='center', color='white' if i==0 else 'black', fontsize=5.5)

ax.set_ylabel('% Neurons', **TEXT_KWARGS)
# 【强制修改 3】
ax.set_xticks(x_pos)
ax.set_xticklabels(groups, rotation=-45, ha='left', rotation_mode='anchor', **TEXT_KWARGS)
#plt.setp(ax.get_xticklabels(), rotation=-45, ha='left', rotation_mode='anchor')


add_bracket_to_bar(ax, x_pos, 100, _gee_place_p(combined_df, 'deep')) 


# ==========================================
# Row 6: Superficial Sub-population Stats
# ==========================================
ax7_1 = fig.add_subplot(gs[6, 0])
ax7_2 = fig.add_subplot(gs[6, 1])
ax7_3 = fig.add_subplot(gs[6, 2])
ax7_4 = fig.add_subplot(gs[6, 3])
ax7_5 = fig.add_subplot(gs[6, 4])
ax7_6 = fig.add_subplot(gs[6, 5])
ax7_7 = fig.add_subplot(gs[6, 6:8])
axis_sup = [ax7_1, ax7_2, ax7_3, ax7_4, ax7_5, ax7_6]

auto_legend_text.append("\nSuperficial Sub-population:")

for i, (var, title) in enumerate(zip(variables, titles)):
    control_superficial = combined_df[(combined_df['group_ani'] == 'control') & (combined_df['sub_population'] == 'superficial')][var].dropna()
    exp_superficial = combined_df[(combined_df['group_ani'] == 'exp') & (combined_df['sub_population'] == 'superficial')][var].dropna()
    
    test_name_sup, stat_sup, p_values = choose_stat_test(control_superficial, exp_superficial, title, "Control_superficial", "Exp_superficial")
    
    # 1. Format the text for Neuron
    n_ctrl = len(control_superficial)
    n_exp = len(exp_superficial)
    p_str = f"p = {p_values:.4g}" if p_values >= 0.0001 else "p < 0.0001"
    
    legend_line = (f"For {title}: {test_name_sup}, statistic = {stat_sup:.3f}, {p_str} "
                   f"(CR;DTA- n = {n_ctrl} cells, CR;DTA+ n = {n_exp} cells).")
    auto_legend_text.append(legend_line)

    subset_df = combined_df[combined_df['depth'] == 'superficial'].dropna(subset=[var])
    plot_elegant_comparison(axis_sup[i], subset_df, 'group_ani', var, p_values, title)
    
# -- Stacked Bar (Superficial) --
ax = ax7_7
counts = [[round(control_place_superficial/len(control_df_superficial),2)*100, round(control_non_place_superficial/len(control_df_superficial),2)*100], 
          [round(exp_place_superficial/len(exp_df_superficial),2)*100, round(exp_non_place_superficial/len(exp_df_superficial),2)*100]]
percentages = counts

# 1. Run the Chi-Square Test
contingency_sup = [
    [control_place_superficial, control_non_place_superficial],
    [exp_place_superficial, exp_non_place_superficial]
]
chi2_stat_sup, p_value_superficial, dof_sup, expected_sup = chi2_contingency(contingency_sup)
print(f"Superficial Proportion Chi-Square: stat={chi2_stat_sup:.3f}, p={p_value_superficial:.4f}")

# 2. Add to Auto-Legend
p_str_bar_sup = f"p = {p_value_superficial:.4g}" if p_value_superficial >= 0.0001 else "p < 0.0001"
bar_legend_sup = (f"Proportion of Place vs Non-place cells (Superficial): Pearson's Chi-square test, \chi^2 = {chi2_stat_sup:.3f}, {p_str_bar_sup} "
                  f"(CR;DTA- n = {len(control_df_superficial)} cells, CR;DTA+ n = {len(exp_df_superficial)} cells).")
auto_legend_text.append(bar_legend_sup)

bars = []
for i in range(len(counts[0])):
    bar = ax.bar(x_pos, [counts[j][i] for j in range(len(counts))], 
                 bottom=[sum(counts[j][:i]) for j in range(len(counts))], 
                 color=[colors[i] if j == 0 else colors1[i] for j in range(len(counts))], 
                 width=bar_width, edgecolor='white', linewidth=0.8)
    bars.append(bar)

for i, bar_group in enumerate(bars):
    cell_type = "Place cell" if i == 0 else "Non\nplace cell"
    for j, bar in enumerate(bar_group):
        ax.text(bar.get_x() + bar.get_width()/2, sum(counts[j][:i]) + bar.get_height()/2, 
                f'{percentages[j][i]:.0f}%\n({cell_type})', 
                ha='center', va='center', color='white' if i==0 else 'black', fontsize=5.5)

ax.set_ylabel('% Neurons', **TEXT_KWARGS)
# 【强制修改 4】
ax.set_xticks(x_pos)
ax.set_xticklabels(groups, rotation=-45, ha='left',  **TEXT_KWARGS)
#plt.setp(ax.get_xticklabels(), rotation=-45, ha='left', rotation_mode='anchor')

#sns.despine(ax=ax, trim=True, offset=3)
add_bracket_to_bar(ax, x_pos, 100, _gee_place_p(combined_df, 'superficial'))

# ==========================================
# 结尾排版与保存
# ==========================================
fig.subplots_adjust(top=0.96, bottom=0.06, left=0.08, right=0.96, hspace=0.6, wspace=0.8)
# 保存为 PDF 格式，方便用 Adobe Illustrator 打开和自由编辑（线条、颜色、字体均可改）
plt.savefig(r'/Users/sachuriga/Desktop/Projects/CR_CA1_paper/Figures_neuron_report_raw/deep_superficial_comparison.pdf', 
            transparent=True, 
            bbox_inches='tight')

plt.show()

In [ ]:
auto_legend_text

## Mixed-effects re-analysis (addressing reviewer's pseudoreplication concern)

审稿人攻击的本质是 **pseudoreplication(伪重复)**:我们把 cell / recording 当成独立样本做 t-test 或
Mann-Whitney,但同一只动物里的 cell 是相关的,真正的独立单位是 **animal**。Mixed-effects model 的做法
就是把 **genotype 当 fixed effect**,把 **animal 放成 random effect(随机截距)**,让模型知道"这些 cell
来自同一只鼠",从而不夸大自由度。

- **连续指标**(info rate, sparsity, field size, rate, selectivity, stability):
  线性混合模型 (LMM) `value ~ genotype + (1 | animal)`,用 `statsmodels.MixedLM`。
  右偏的指标先做 log 变换以满足残差近似正态。
- **place cell 比例**(二分类):cluster-robust logistic regression (GEE),按 animal 聚类,
  等价于把动物相关性纳入标准误,给出频率派 p 值。报告 odds ratio。


In [ ]:
# ============================================================
# Mixed-effects / cluster-robust re-analysis
# Fixed effect: genotype (group_ani);  Random effect: animal (random intercept)
# 依赖上面已经构建好的 combined_df (含 group_ani, sub_population, animal_id, 各指标)
# ============================================================
import statsmodels.formula.api as smf
import statsmodels.api as sm
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

mdf = combined_df.copy()
mdf['animal_id'] = mdf['animal_id'].astype(str)
mdf['group_ani'] = pd.Categorical(mdf['group_ani'], categories=['control', 'exp'])

# place cell 定义(与上方饼图/堆叠图保持一致)
mdf['matlab_maxfsize'] = pd.to_numeric(mdf['matlab_maxfsize'], errors='coerce')
mdf['Information_content_rate'] = pd.to_numeric(mdf['Information_content_rate'], errors='coerce')
mdf['is_place'] = ((mdf['h_0_place_cell'] == 1) &
                   (mdf['Information_content_rate'] >= 1.68) &
                   (mdf['matlab_maxfsize'] >= 20)).astype(int)

variables = ['Information_content_rate', 'Sparsity', 'Field_size',
             'Averate_rate', 'Selectivity', 'stability_ma']
titles = ['Information content rate', 'Sparsity', 'Max Field Size',
          'Firing Rate', 'Selectivity', 'Stability']
# 右偏指标:LMM 前做 log 变换,使残差近似正态(stability/sparsity 通常已近似对称,不变换)
log_vars = {'Information_content_rate', 'Field_size', 'Averate_rate', 'Selectivity'}


def lmm_test(data, var, log=False):
    """LMM: value ~ genotype + (1|animal). 返回 (beta_exp, p, n_cells, n_animals, note)."""
    d = data.copy()
    d['y'] = pd.to_numeric(d[var], errors='coerce')
    d = d.dropna(subset=['y'])
    note = ''
    if log:
        d = d[d['y'] > 0]
        d['y'] = np.log(d['y'])
        note = 'log'
    m = smf.mixedlm("y ~ group_ani", d, groups=d['animal_id']).fit(reml=True)
    key = 'group_ani[T.exp]'
    return m.params[key], m.pvalues[key], len(d), d['animal_id'].nunique(), note


def gee_logit(data):
    """Cluster-robust logistic (GEE), cluster=animal. 返回 (beta, p, OR, n_cells, n_animals)."""
    d = data.dropna(subset=['is_place']).copy()
    g = smf.gee("is_place ~ group_ani", "animal_id", d,
                family=sm.families.Binomial(),
                cov_struct=sm.cov_struct.Exchangeable()).fit()
    key = 'group_ani[T.exp]'
    return g.params[key], g.pvalues[key], np.exp(g.params[key]), len(d), d['animal_id'].nunique()


rows = []
for sp in ['deep', 'superficial']:
    sub_all = mdf[mdf['sub_population'] == sp]
    for var, title in zip(variables, titles):
        beta, p, nc, na, note = lmm_test(sub_all, var, log=(var in log_vars))
        rows.append(dict(sub_population=sp, metric=title,
                         model='LMM' + (' (log)' if note else ''),
                         beta_exp=round(beta, 3), p=round(p, 4),
                         n_cells=nc, n_animals=na))
    beta, p, OR, nc, na = gee_logit(sub_all)
    rows.append(dict(sub_population=sp, metric='Place-cell proportion',
                     model='GEE-logit', beta_exp=round(beta, 3), p=round(p, 4),
                     n_cells=nc, n_animals=na, OR=round(OR, 3)))

res_mixed = pd.DataFrame(rows)
pd.set_option('display.width', 160)
print(res_mixed.to_string(index=False))

# 自动生成 rebuttal 用文字
print("\n--- Rebuttal-ready sentences ---")
for r in rows:
    star = '***' if r['p'] < 0.001 else '**' if r['p'] < 0.01 else '*' if r['p'] < 0.05 else 'n.s.'
    extra = f", OR = {r['OR']}" if 'OR' in r and pd.notna(r.get('OR')) else ''
    print(f"[{r['sub_population']}] {r['metric']}: {r['model']}, "
          f"β(CR;DTA+) = {r['beta_exp']}{extra}, p = {r['p']} {star} "
          f"(n = {r['n_cells']} cells from {r['n_animals']} mice).")


## In-field vs. out-of-field firing rate (place-coding accuracy / spatial signal-to-noise)

要回答"CR;DTA+ 的 place cell 空间编码准确性是否受损",标准做法是比较 **place field 内 vs 外的发放率**:

- **In-field rate** = field 内总 spikes / field 内总停留时间 (Hz)
- **Out-field rate** = field 外(已访问 bin)总 spikes / field 外总停留时间 (Hz)
- **In/out ratio (spatial signal-to-noise)** = in_field_rate / out_field_rate
  → 比值越高 = field 越"干净"、空间编码越锐利;若 CR;DTA+ 比值下降,说明 place coding 精度受损。

实现完全复用现有 pipeline:`load_speed_fromNWB` → `pos2speed`(speed filter)→ `Speed_filtered_spikes`
→ `SpatialMap.rate_map`(检测 field 的平滑图)+ 一张 `smoothing=0` 的原始 spike/occupancy 图(算真实 Hz)
→ `separate_fields_by_laplace` 出 field mask。

> ⚠️ **这段需要 NWB 原始位置数据(`S:\Sachuriga\nwb\test4neo`),要在能访问该盘的机器上运行**;
> 本机(Mac)读不到 NWB。算完得到的 per-cell 表,再丢进上面的 LMM 混合模型比较(genotype fixed,
> animal random),才能避免 pseudoreplication。


In [ ]:
# ============================================================
# Compute in-field vs out-of-field firing rate per cell
# 复用现有 pipeline。需在能访问 NWB (S:\Sachuriga\nwb\test4neo) 的机器上运行。
# 依赖 combined_df (含 session_id, spike_times, animal_id, group_ani, sub_population)
# ============================================================
import os
os.chdir(r'/Users/sachuriga/Desktop/code/nwb4fp/src')   # Windows 上改成 ...\quattrocolo-nwb4fp\src
import numpy as np
import pandas as pd
import pynapple as nap
import nwb4fp.analyses.maps as mapp
from nwb4fp.analyses.data import pos2speed, load_speed_fromNWB, Speed_filtered_spikes
from nwb4fp.analyses.fields import separate_fields_by_laplace

base_nwb_folder = r"S:\Sachuriga\nwb\test4neo"

# --- 参数 (与 place-cell 分类口径保持一致) ---
BIN_SIZE        = 0.05    # field 检测分辨率 (1x1 box -> 20x20 bins)
SMOOTHING       = 0.05    # 平滑图(检测 field 用)
FIELD_THRESHOLD = 0.1     # separate_fields_by_laplace 的 laplacian 阈值
MIN_FIELD_BINS  = 4       # 小于这个 bin 数的 field 丢弃 (按需调整)
USE_PRIMARY_ONLY = False  # True: 只用最大那个 field(label==1);False: 所有 field 合并


def infield_outfield_rates(x, y, t, spike_times):
    """返回 (in_rate, out_rate, ratio, peak_rate, n_fields, field_frac)。无 field 时 ratio=nan。"""
    sm = mapp.SpatialMap(box_size=[1.0, 1.0], bin_size=BIN_SIZE, smoothing=SMOOTHING)
    rate_map = sm.rate_map(x, y, t, spike_times)                       # 平滑图,用于检测 field

    labels = separate_fields_by_laplace(rate_map, threshold=FIELD_THRESHOLD,
                                        minimum_field_area=MIN_FIELD_BINS)
    n_fields = int(labels.max())
    peak_rate = float(np.nanmax(rate_map))

    # 原始(未平滑)spike 计数 与 停留时间 -> 真实 Hz
    raw = mapp.SpatialMap(box_size=[1.0, 1.0], bin_size=BIN_SIZE, smoothing=0)
    spk_cnt = raw.spike_map(x, y, t, spike_times, mask_zero_occupancy=False)
    occ_t   = raw.occupancy_map(x, y, t, mask_zero_occupancy=False)

    visited = occ_t > 0
    if USE_PRIMARY_ONLY:
        infield = (labels == 1)
    else:
        infield = (labels >= 1)
    outfield = visited & (~infield)

    if infield.sum() == 0 or occ_t[infield].sum() == 0:
        return np.nan, np.nan, np.nan, peak_rate, n_fields, 0.0

    in_rate  = spk_cnt[infield].sum()  / occ_t[infield].sum()
    out_rate = (spk_cnt[outfield].sum() / occ_t[outfield].sum()
                if occ_t[outfield].sum() > 0 else np.nan)
    ratio = in_rate / out_rate if (out_rate and out_rate > 0) else np.nan
    field_frac = infield.sum() / visited.sum()       # field 占已访问空间比例
    return in_rate, out_rate, ratio, peak_rate, n_fields, field_frac


# --- 遍历 session,逐 unit 计算 ---
records = []
for sid in np.unique(combined_df['session_id']):
    try:
        npdata = nap.load_file(fr"{base_nwb_folder}/{sid}")
        pos_cord = load_speed_fromNWB(npdata['XY_mid_brain'])
        raw_pos, comb, mask, *_ = pos2speed(pos_cord[:, 0], pos_cord[:, 1], pos_cord[:, 2],
                                            filter_speed=True, min_speed=0.05)
    except Exception as e:
        print(f"[skip] {sid}: {e}")
        continue

    sub_tab = combined_df[combined_df['session_id'] == sid]
    for idx, row in sub_tab.iterrows():
        st = Speed_filtered_spikes(np.asarray(row['spike_times']), pos_cord[:, 0], mask)
        in_r, out_r, ratio, peak, nf, ff = infield_outfield_rates(
            comb[:, 1], comb[:, 2], comb[:, 0], st)
        records.append(dict(
            orig_index=idx, session_id=sid,
            animal_id=str(row['animal_id']), group_ani=row['group_ani'],
            sub_population=row['sub_population'],
            is_place=int(row.get('is_place', np.nan)) if 'is_place' in row else np.nan,
            in_field_rate=in_r, out_field_rate=out_r,
            in_out_ratio=ratio, peak_rate=peak,
            n_fields=nf, field_fraction=ff))

field_df = pd.DataFrame(records)
field_df['log_in_out_ratio'] = np.log(field_df['in_out_ratio'].where(field_df['in_out_ratio'] > 0))
# 保存以便复用 / 离线在 Mac 上做统计
field_df.to_pickle(r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/infield_outfield_rates.pkl")
print("computed cells:", len(field_df))
print(field_df.groupby(['sub_population', 'group_ani'])[
    ['in_field_rate', 'out_field_rate', 'in_out_ratio']].median())


In [ ]:
# ============================================================
# Mixed-effects comparison of in/out-field rates (control vs exp, per layer)
# 可离线在 Mac 上跑:从保存的 pkl 读 field_df
# ============================================================
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
import warnings; warnings.filterwarnings('ignore')

# field_df = pd.read_pickle(r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/infield_outfield_rates.pkl")
fdf = field_df.copy()
fdf['group_ani'] = pd.Categorical(fdf['group_ani'], categories=['control', 'exp'])

# 只在 place cell 上比较空间编码精度(有 field 的细胞);如需全 pyramidal,去掉这行
fdf_pc = fdf[(fdf['is_place'] == 1) & fdf['in_out_ratio'].notna()].copy()


def lmm_field(data, var, log=False):
    d = data.copy()
    d['y'] = pd.to_numeric(d[var], errors='coerce')
    d = d.dropna(subset=['y'])
    if log:
        d = d[d['y'] > 0]; d['y'] = np.log(d['y'])
    m = smf.mixedlm("y ~ group_ani", d, groups=d['animal_id']).fit(reml=True)
    k = 'group_ani[T.exp]'
    return m.params[k], m.pvalues[k], len(d), d['animal_id'].nunique()

metrics = [('in_field_rate', True), ('out_field_rate', True),
           ('in_out_ratio', True), ('peak_rate', True), ('field_fraction', False)]

rows = []
for sp in ['deep', 'superficial']:
    sub = fdf_pc[fdf_pc['sub_population'] == sp]
    for var, log in metrics:
        try:
            beta, p, nc, na = lmm_field(sub, var, log=log)
        except Exception as e:
            beta, p, nc, na = np.nan, np.nan, len(sub), sub['animal_id'].nunique()
        med_c = sub[sub['group_ani'] == 'control'][var].median()
        med_e = sub[sub['group_ani'] == 'exp'][var].median()
        rows.append(dict(sub_population=sp, metric=var, model='LMM' + (' log' if log else ''),
                         median_ctrl=round(med_c, 3), median_exp=round(med_e, 3),
                         beta_exp=round(beta, 3), p=round(p, 4), n_cells=nc, n_animals=na))

res_field = pd.DataFrame(rows)
pd.set_option('display.width', 180)
print(res_field.to_string(index=False))

print("\n--- Rebuttal sentences ---")
for r in rows:
    star = '***' if r['p'] < 0.001 else '**' if r['p'] < 0.01 else '*' if r['p'] < 0.05 else 'n.s.'
    print(f"[{r['sub_population']}] {r['metric']}: LMM, β(CR;DTA+) = {r['beta_exp']}, p = {r['p']} {star} "
          f"(median ctrl={r['median_ctrl']} vs exp={r['median_exp']}; "
          f"n = {r['n_cells']} place cells from {r['n_animals']} mice).")
